In [ ]:
!pip install optuna

In [ ]:
!pip install optuna.integration

In [6]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from optuna.integration import LightGBMPruningCallback
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import os
import json


In [ ]:
# ===============================
# Chargement des fichiers CSV
# ===============================

train_df = pd.read_csv("train_features_cleaned.csv", index_col=0)
test_df = pd.read_csv("test_features_cleaned.csv", index_col=0)
target_df = pd.read_csv("train_target_corr.csv", index_col=0)


# ===============================
# 1. Matrices finales
# ===============================
X = train_df.astype(float).values
y = target_df.values.ravel()
X_test = test_df.astype(float).values

print(f"Taille de X : {X.shape}")

# ===============================
# 2. Définition de la fonction Objective pour Optuna
# ===============================
def objective(trial):
    # 1. Grille de recherche des Hyperparamètres
    param = {
        'objective': 'multiclass',
        'num_class': len(np.unique(y)),
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,

        # Vitesse et structure
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 30, 150),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),

        # Régularisation (Pour éviter d'apprendre par coeur)
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 6, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 6, log=True),

        # Bagging et Feature sampling (Robustesse)
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.95),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),

        'n_jobs': -1
    }

    # 2. Validation Croisée (3 Folds pour gagner du temps)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_f, y_train_f = X[train_idx], y[train_idx]
        X_val_f, y_val_f = X[val_idx], y[val_idx]

        dtrain = lgb.Dataset(X_train_f, label=y_train_f)
        dval = lgb.Dataset(X_val_f, label=y_val_f, reference=dtrain)

        # Le Pruning callback permet à Optuna d'abandonner un test s'il est mauvais dès les premiers arbres
        pruning_callback = LightGBMPruningCallback(trial, 'multi_logloss')

        model = lgb.train(
            param,
            dtrain,
            num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                pruning_callback
            ]
        )

        # Évaluation
        cv_scores.append(model.best_score['valid_0']['multi_logloss'])


    return np.mean(cv_scores)

# ===============================
# 3. Lancement de la recherche Optuna
# ===============================
print("Lancement d'Optuna (Recherche des meilleurs hyperparamètres)...")
# On crée une étude orientée vers la maximisation du score
study = optuna.create_study(direction='minimize',
                            study_name="LGBM_Tuning",
                            storage="sqlite:////content/drive/MyDrive/Data/optuna_study.db",
                            load_if_exists=True)

# nbr essais 
study.optimize(objective, n_trials=30)


print(f"Meilleur Multi LogLoss (Validation 3-Folds) : {study.best_value:.5f}")
print("Meilleurs Hyperparamètres trouvés :")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")


SAVE_PATH = "/content/drive/MyDrive/Data/optuna_results"
os.makedirs(SAVE_PATH, exist_ok=True)

# Tous les résultats
df_results = study.trials_dataframe()
df_results.to_csv(os.path.join(SAVE_PATH, "all_trials.csv"), index=False)

print("Tous les trials sauvegardés")



best_params = study.best_params
best_value = study.best_value

with open(os.path.join(SAVE_PATH, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=4)

with open(os.path.join(SAVE_PATH, "best_score.txt"), "w") as f:
    f.write(f"Best multi_logloss: {best_value:.6f}")

print("Meilleurs hyperparamètres sauvegardés")

summary = {
    "best_score": best_value,
    "n_trials": len(study.trials),
    "completed_trials": len([t for t in study.trials if t.state.name == "COMPLETE"]),
    "pruned_trials": len([t for t in study.trials if t.state.name == "PRUNED"])
}

with open(os.path.join(SAVE_PATH, "study_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Résumé sauvegardé")
